# Phase 3A — one-day meta-learning feasibility pilot

Run this notebook in VS Code with a connected **Colab GPU** kernel. It mounts Google Drive, clones the public project into a dedicated Drive folder, verifies CUDA, and runs the locked Phase 3A experiment. No terminal is required.

**Before you begin:** push the current Phase 3A commits from your Mac to GitHub. This notebook downloads that exact public repository state.

The final cell opens the held-out TEM-1 test positions exactly once. After a completed run, do not run that cell again.

## 1. Confirm the connected runtime

The output must show `cuda` and an NVIDIA GPU (an L4 is suitable). If it does not, use **Select Kernel → Colab → New Colab Server**, choose GPU, then rerun this cell.

In [ ]:
import platform
import subprocess
import sys

import torch

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. Reconnect this notebook to a Colab GPU first.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
subprocess.run(['nvidia-smi'], check=True)

## 2. Mount Drive and obtain the project

All data, model checkpoints, and results are stored under `MyDrive/esm2-phase3a`, so they survive a Colab runtime restart. On later sessions, this cell updates the existing clone instead of making a second copy.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO_DIR = DRIVE_ROOT / 'esm2-phase3a'
REPO_URL = 'https://github.com/AnanyaSSadana/esm2-mutation-effects.git'

if (REPO_DIR / '.git').is_dir():
    print(f'Updating existing clone: {REPO_DIR}')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename it in Drive, then rerun this cell.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

print(subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip())
print(REPO_DIR)

## 3. Install the project and verify the checkout

This installs the project into the active Colab kernel. It does not copy local Mac files; the repository must already contain the Phase 3A code.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/run_phase3a.py', '--help'], cwd=REPO_DIR, check=True)
print('Setup complete.')

## 4. Launch the locked Phase 3A pilot

This command downloads the authoritative ProteinGym pilot data, generates zero-shot scores, writes the strict 401-label target manifest and protocol lock, trains the standard/pooled/Reptile comparators, and evaluates the held-out target positions once. Expect several GPU hours.

If this cell fails **before** `test_predictions.csv` exists, reconnect the GPU and rerun this cell. If it succeeds, do not rerun it.

In [ ]:
from pathlib import Path

RUN_DIR = REPO_DIR / 'results' / 'phase3a_pilot' / 'run'
if (RUN_DIR / 'test_predictions.csv').exists() or (RUN_DIR / 'summary.json').exists():
    raise RuntimeError(
        'This pilot already has target-test output. Do not rerun it; use the next cell to inspect results.'
    )

subprocess.run(
    [sys.executable, 'scripts/run_phase3a.py', '--device', 'cuda', '--microbatch-size', '16'],
    cwd=REPO_DIR,
    check=True,
)

## 5. Inspect completed results

Run only after the launch cell completes successfully. Save or download the displayed `summary.json` before making any interpretation; the result is a single-split feasibility study, not a generalization claim.

In [ ]:
import json
import pandas as pd

summary_path = RUN_DIR / 'summary.json'
if not summary_path.exists():
    raise FileNotFoundError('No completed summary yet. Finish the launch cell first.')

summary = json.loads(summary_path.read_text())
display(pd.DataFrame(summary['results']))
print(f'Saved summary: {summary_path}')
print('Saved predictions:', RUN_DIR / 'test_predictions.csv')